<b><font size="6" color="#E8800A">Week 11d: Combining models for regression</font></b><br>
<b><font size="4">Training-only selection, one sealed test partition</font></b><br>

Voting fixes the combining rule. Stacking and blending fit that rule on predictions from rows the members did not train on.

<div class="alert alert-block alert-info">

# TOC<a class="anchor" id="toc"></a>
* [<font color='#E8800A'>Setup</font>](#setup)
* [<font color='#E8800A'>The model pool</font>](#pool)
* [<font color='#E8800A'>Voting</font>](#vote)
* [<font color='#E8800A'>Voting settings</font>](#vote_settings)
* [<font color='#E8800A'>Stacking</font>](#stack)
* [<font color='#E8800A'>Stacking settings</font>](#stack_settings)
* [<font color='#E8800A'>Blending</font>](#blend)
* [<font color='#E8800A'>Blending settings</font>](#blend_settings)
* [<font color='#E8800A'>Selection of the combiners</font>](#sel2)
* [<font color='#E8800A'>The combiner winner</font>](#combiner_winner)
* [<font color='#E8800A'>The course board</font>](#board)
* [<font color='#E8800A'>Key takeaways</font>](#takeaways)
* [<font color='#E8800A'>Exit ticket</font>](#exit)
* [<font color='#E8800A'>References</font>](#references)

</div>

# <font color='#E8800A'>Setup</font> <a class="anchor" id="setup"></a>
[Back to TOC](#toc)

Run the boosting notebook for this task first. Its saved selection must match these data, recipe, split and folds; a missing or stale file stops execution before a combiner is fitted.

In [ ]:
# The notebook folder holds preprocessing.py and course_helpers.py, so it joins the import path first.
import sys
from pathlib import Path
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

# Libraries for data handling, figures and parallel work.
import os
import json
import time
import warnings
import joblib
from itertools import combinations
import sklearn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# scikit-learn: the models, the validation tools and the metrics used below.
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import VotingRegressor, RandomForestRegressor, BaggingRegressor, StackingRegressor
from sklearn.linear_model import Lasso, BayesianRidge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import ParameterGrid, RandomizedSearchCV, cross_validate, train_test_split, KFold, cross_val_predict
from sklearn.preprocessing import FunctionTransformer
# Progress bars, then the course helpers: the figure palette, the model log and the scoreboard.
from tqdm.auto import tqdm
from course_helpers import PLOT_BLUE, PLOT_ORANGE, Scoreboard, record, results_before, result_context
# The fold-owned preparation, stacking and blending classes written for this course.
from preprocessing import FoldStacking, FoldBlending, LockedMemory, fold_owned_members, unprepared_boosting
from matplotlib_inline.backend_inline import set_matplotlib_formats

# One seed for every random choice, then the figure style: white grid, 150 dpi, vector output.
random_state = 33
np.random.seed(random_state)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 150
set_matplotlib_formats("svg")
# The exact-message filter also reaches the parallel workers through the environment; the library message is expected, because a validation fold can hold a category the training rows lack.
os.environ["PYTHONWARNINGS"] = "ignore:Found unknown categories:UserWarning"
warnings.filterwarnings("ignore", message="Found unknown categories.*during transform", category=UserWarning)
print("scikit-learn", sklearn.__version__)
from preprocessing import load_boosting_selection

from preprocessing import (
    CachedSearchCV,
    FoldPreprocessor,
    PreparedEstimator,
    load_recipe,
    outer_folds,
    outer_splitter,
    prepared_folds,
)
regression_recipe_log = "../../logs/week_04_feature_work_regression_log.json"


__Step 1:__ Load the data and prepare the same training folds.

In [ ]:
# The cars4you table with the feature recipe of the feature-work notebook.
recipe_log = "../../logs/week_04_feature_work_regression_log.json"
frame, recipe = load_recipe("../../data/interim/cars4you.csv", recipe_log)

# Features and target, then the test rows are sealed once: 20 percent of the cars, and no choice below reads them.
features = frame.loc[:, recipe["categorical"] + recipe["numeric"]].copy()
target = frame["price"]
X_train, X_test, y_train, y_test = train_test_split(features, target,
    test_size=0.2, random_state=random_state)
# `prepared` and `raw_prepared` hold, for each of the ten folds, the rows to fit (`X_fit`, `y_fit`) and the rows to score (`X_score`, `y_score`): prepared with the recipe and left raw.
preparation = FoldPreprocessor(recipe)
folds = outer_splitter("regression", random_state=random_state)
prepared = prepared_folds(preparation, list(folds.split(X_train, y_train)), X_train, y_train)
raw_prepared = prepared_folds(FunctionTransformer(), list(folds.split(X_train, y_train)), X_train, y_train)
# The recipe is fitted once on all the training rows. Every model fitted on all the rows learns from `Z_train` and is scored on `Z_test`, the same prepared rows, so no model refits the recipe.
full_prep = clone(preparation).fit(X_train, y_train)
Z_train = full_prep.transform(X_train)
Z_test = full_prep.transform(X_test)
# `model_log` collects every logged result in order; `results` is the folder of saved results and of this notebook's fitted winner.
model_log = []
results_dir = "../../results"
print("training folds:", folds.get_n_splits())

__Step 2:__ Write the earlier learners once and create the shared member cache.

In [ ]:
taught_learners = {
    "Lasso, log price": TransformedTargetRegressor(regressor=Lasso(alpha=0.0001, max_iter=10000), func=np.log, inverse_func=np.exp),
    "Bayesian ridge": BayesianRidge(lambda_1=0.0001, alpha_1=100000, max_iter=2),
    "KNN": KNeighborsRegressor(n_neighbors=5, p=1, weights="distance"),
    "pruned tree, log price": TransformedTargetRegressor(regressor=DecisionTreeRegressor(ccp_alpha=0.0001, criterion="absolute_error", max_depth=None, max_features=None, min_samples_leaf=5, min_samples_split=2, splitter="best", random_state=random_state), func=np.log, inverse_func=np.exp),
    "bagged trees": BaggingRegressor(estimator=DecisionTreeRegressor(random_state=33), n_estimators=105, n_jobs=4, random_state=33),
    "random forest": RandomForestRegressor(min_samples_leaf=2, n_jobs=4, random_state=33),
}

<div class="alert alert-block alert-info">
<code>LockedMemory(location=None, backend="local", mmap_mode=None, compress=False, verbose=1, backend_options=None)</code>

A course class in `preprocessing.py`, beside this notebook.

**Definition:**
A joblib `Memory` with a lock file keyed by the function and arguments. The workers wait for the first computation, then read its cache entry.

**Main Parameters:**
- `location`: the folder of the cache; `None` switches the cache off.
- The other parameters are those of joblib `Memory`.

**Main Methods:**
- `cache(func)`: the cached, locked version of `func`.

**Differences from scikit-learn:**
scikit-learn uses joblib [`Memory`](https://joblib.readthedocs.io/en/latest/user_guide/memory.html), for example in `Pipeline(memory=...)`, which allows concurrent computation of a missing entry. `LockedMemory` prevents repeated preprocessing fits across workers. Without the `filelock` package, locking is skipped: duplicate computation remains possible, with the same result.
</div>

__Step 3:__ Compare the earlier model labels with their explicit constructors.

In [ ]:
# Compare the explicit constructors with the stored labels of the earlier models.
for (name, learner), week in zip(list(taught_learners.items())[:4], range(6, 10)):
    stored = json.loads(Path(results_dir, f"week_{week:02d}.json").read_text(encoding="utf-8"))["regression"]["model"]
    print(f"{name} | week_{week:02d}: {stored}")
    print("constructor:", learner)
for name, learner in list(taught_learners.items())[4:]:
    print(f"{name} | best evaluated in the previous week")
    print("constructor:", learner)
member_memory = LockedMemory(".ensemble_member_cache", verbose=0)

<div class="alert alert-block alert-info">
<code>load_boosting_selection(path, *, task, context, fold_fingerprint, n_splits)</code>

A helper in `preprocessing.py`, beside this notebook.

**Definition:** Loads the saved boosting winner only when its evidence matches this notebook.

**Main Parameters:** `path` names the selection; `task`, `context`, `fold_fingerprint` and `n_splits` describe the current data and folds.

**Returns:** Selected settings, fold scores and the fitted winner.

**Differences from scikit-learn:** [joblib.load](https://joblib.readthedocs.io/en/latest/generated/joblib.load.html) restores a file. This course helper also checks the task, data, recipe, folds, array shapes and fitted-model checksum. A mismatch asks you to rerun boosting; it never starts another boosting search.
</div>

__Step 4:__ Validate and load the boosting selection before fitting a combiner.

In [ ]:
# Fingerprint the data, recipe, split and exact outer rows.
selection_context = result_context(
    "../../data/interim/cars4you.csv", recipe_log, y_train.index.tolist(), y_test.index.tolist(),
    outer_folds=outer_folds(folds, X_train, y_train))
fold_fingerprint = joblib.hash(outer_folds(folds, X_train, y_train))
selection = load_boosting_selection(Path(results_dir) / "boosting" / "week_11_regression_selection.joblib",
    task="regression", context=selection_context, fold_fingerprint=fold_fingerprint, n_splits=folds.get_n_splits())
# Restore settings selected independently on each outer training fold.
boost_settings = selection["boost_settings"]
boost_fold_settings = selection["boost_fold_settings"]
boost_nested_scores = selection["boost_nested_scores"]
boost_nested_validation = np.asarray(selection["boost_nested_validation"])
boost_nested_train = np.asarray(selection["boost_nested_train"])
boost_test = selection["boost_test"]
boost_label = selection["boost_label"]
boosting_winner = selection["boosting_winner"]
boost_baseline_cv = selection["baseline_cv"]
boost_baseline_test = selection["baseline_test"]
# Rebuild the unfitted member; its fitted all-training winner remains separate.
boosting_member = clone(boost_settings["estimator"]).set_params(**{key.split("estimator__", 1)[1]: value for key, value in boost_settings.items() if key.startswith("estimator__")})
inner = KFold(n_splits=3, shuffle=True, random_state=random_state)
print("Loaded boosting winner:", boost_label)
print("Outer-training selections:", len(boost_fold_settings))

# <font color='#E8800A'>The model pool</font> <a class="anchor" id="pool"></a>
[Back to TOC](#toc)

Voting, stacking and blending draw their members from seven models: the six earlier models and the boosting winner. Every combiner tries every combination of two to seven of them.

<div class="alert alert-block alert-info">
<code>fold_owned_members(preprocessor, estimators, memory=None)</code>

A helper in `preprocessing.py`, beside this notebook.

**Definition:**
Wraps `(name, learner)` pairs as members that prepare their own raw rows: `CachedPreparedMember`, a `PreparedEstimator` subclass. With memory and fixed learner randomness, `fit` is lazy: predictions for identical fit and score rows are computed once, then cached.

**Main Parameters:**
- `preprocessor`: the unfitted recipe; a clone of it is fitted on the training rows of each member.
- `estimators`: a list of `(name, learner)` pairs.
- `memory`: an optional joblib `Memory` that holds the cached predictions.

**Returns:**
A list of `(name, member)` pairs. `CalibratedClassifierCV` is separated into base learner and calibration method so calibration owns the raw training split. Boosted learners whose bases already prepare raw rows receive the raw frame; others receive a recipe clone.

**Differences from scikit-learn:**
[`VotingRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.VotingRegressor.html) and [`StackingRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingRegressor.html) fit supplied learners on supplied rows. Here members own preparation and cache predictions by row identity, allowing combinations to reuse them without refitting.
</div>

__Step 5:__ Write the pool of seven models, wrap each so that it prepares its own raw rows, and name the kind of each.

In [ ]:
# The seven models the combiners draw from: the six earlier models and the boosting winner.
members = list(taught_learners.items()) + [("boosting winner", boosting_member)]
# The same members, each wrapped so that it prepares its own raw rows; `member_memory` stores their predictions for reuse.
raw_members = fold_owned_members(preparation, members, memory=member_memory)
# The kind of each model, used below to count how many kinds a combination mixes.
kinds = {'Lasso, log price': 'linear', 'Bayesian ridge': 'linear', 'KNN': 'nearest neighbours', 'pruned tree, log price': 'single tree', 'bagged trees': 'tree ensemble', 'random forest': 'tree ensemble', 'boosting winner': 'tree ensemble'}
print("pool:", [name for name, _ in members])
print("kinds:", sorted(set(kinds.values())))

**The pool holds seven models, the six earlier ones and the boosting winner, which is `GradientBoostingRegressor(alpha=0.95, loss='huber', max_depth=5, max_features='sqrt', min_samples_leaf=5, n_estimators=130, random_state=33, subsample=0.8)`.** The tables and figures below call it `boosting winner`; the log and the board name it by its class and these settings. The seven fall into four kinds: linear, nearest neighbours, a single tree and tree ensembles.

# <font color='#E8800A'>Voting</font> <a class="anchor" id="vote"></a>
[Back to TOC](#toc)

A fixed average uses the same combination rule for every row. Regression averages predicted prices. Members retain the settings of the earlier weeks. [Voting guide](https://scikit-learn.org/stable/modules/ensemble.html#voting-classifier)

__Step 6:__ Combine three toy member answers by hand before fitting a vote.

In [ ]:
toy_member_answers = np.array([12., 15., 18.])
print("member prices:", toy_member_answers, "mean:", toy_member_answers.mean())

**The three member prices 12, 15 and 18 average to 15.0.**

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.VotingRegressor.html'>sklearn.ensemble.VotingRegressor(estimators, weights=None, n_jobs=None)</a>

**Definition:**
Fits every member independently on the same rows and averages their predictions.

**Main Parameters:**
- `estimators`: the named members.
- `weights`: multiplier on each member's prediction.
- `n_jobs`: workers for the member fits.

**Main Attributes (after fitting):**
- `estimators_`: the fitted members.
- `named_estimators_`: the fitted members by name.

**Main Methods:**
- `fit(X, y)`: fit each member.
- `predict(X)`: the weighted average.
</div>

The combination rule is fixed by the class: no test column chooses membership or weights.

__Step 7:__ Fit scikit-learn `VotingRegressor` with fresh members and the vote of cached members on the same rows and compare their answers.

In [ ]:
# A small member set and the first fold of raw rows, so the two votes can be compared quickly.
equal_members = [(name, taught_learners[name]) for name in ['Bayesian ridge', 'KNN', 'pruned tree, log price']]
equal_fold = raw_prepared[0]
# scikit-learn's vote: every member is wrapped with the recipe so it prepares its own rows.
started = time.perf_counter()
library_vote = VotingRegressor([(name, PreparedEstimator(FoldPreprocessor(recipe), clone(member))) for name, member in equal_members], n_jobs=4).fit(equal_fold["X_fit"], equal_fold["y_fit"])
library_answer = library_vote.predict(equal_fold["X_score"])
library_vote_seconds = time.perf_counter() - started
# The vote of cached members: the first fit fills the cache, the second, timed, fit reads the member predictions from it.
vote_memory = joblib.Memory(".equal_vote_cache", verbose=0)
VotingRegressor(fold_owned_members(preparation, equal_members, memory=vote_memory), n_jobs=4).fit(equal_fold["X_fit"], equal_fold["y_fit"]).predict(equal_fold["X_score"])
started = time.perf_counter()
cached_vote = VotingRegressor(fold_owned_members(preparation, equal_members, memory=vote_memory), n_jobs=4).fit(equal_fold["X_fit"], equal_fold["y_fit"])
cached_answer = cached_vote.predict(equal_fold["X_score"])
cached_vote_seconds = time.perf_counter() - started
# The two votes must give the same answers; stop here if they differ, and print the largest difference with the two timings.
assert np.array_equal(library_answer, cached_answer), np.abs(library_answer - cached_answer).max()
print("held-out answers are identical:", np.array_equal(library_answer, cached_answer), "| largest difference", np.abs(library_answer - cached_answer).max())
print(f"VotingRegressor {library_vote_seconds:.2f} s, the vote of cached members {cached_vote_seconds:.2f} s")

**The two votes reproduce the same held-out predictions within the stated tolerance.** Caching reuses each member's answers without changing the vote.

__Step 8:__ Declare the member combinations and the weight alternatives before seeing any combination scores.

The seven models give 120 combinations of two to seven members, excluding single-model combiners. Each member's fold predictions are reused across votes, stacks and blends. Combination sweeps use the first three of ten training folds; the boosting and combiner selections use all ten. Weights vary only for the full pool of seven. Rather than refitting [`VotingRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.VotingRegressor.html) for every combination, the vote averages cached predictions from one member fit per fold. The equality check above matches this result to fresh-member fits.

In [ ]:
combiner_folds = raw_prepared[:3]
member_index = [combo for size in range(2, len(members) + 1) for combo in combinations(range(len(members)), size)]
member_values = [[members[i] for i in combo] for combo in member_index]
weight_values = [None] + [tuple(2 if j == i else 1 for j in range(len(members))) for i in range(len(members))]
vote_reference = VotingRegressor(estimators=raw_members, n_jobs=4)

__Step 9:__ Score the VotingRegressor reference on the first three training folds.

In [ ]:
fits = joblib.Parallel(n_jobs=4)(joblib.delayed(clone(vote_reference).fit)(fold["X_fit"], fold["y_fit"]) for fold in combiner_folds)
vote_default_train = [mean_absolute_error(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(combiner_folds, fits)]
vote_default_validation = [mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(combiner_folds, fits)]

__Step 10:__ Fit the VotingRegressor reference on all the training rows.

In [ ]:
vote_default = clone(vote_reference).fit(X_train, y_train)
vote_default_configuration = f"VotingRegressor(estimators=[{', '.join(boost_label if member == 'boosting winner' else member for member, _ in vote_reference.estimators)}]" + ("" if vote_reference.weights is None else f", weights={vote_reference.weights}") + ")"

__Step 11:__ Log the VotingRegressor reference.

In [ ]:
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": vote_default_configuration,
    "train": vote_default_train, "validation": vote_default_validation, "model": vote_default},
    X_test, y_test, "untuned", metric=mean_absolute_error, label="MAE")

# <font color='#E8800A'>Voting settings</font> <a class="anchor" id="vote_settings"></a>
[Back to TOC](#toc)

Each sweep changes one setting of the vote and scores every value on the first three training folds, because every combination refits all of its members on each fold. The section closes on the best configuration actually evaluated.

| Parameter | Meaning |
| --- | --- |
| `estimators` | Members whose answers are combined. |
| `weights` | Multiplier on each member's answer. |

[Parameter reference](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.VotingRegressor.html)

__Step 12:__ Change only voting `estimators` on the fixed training folds, using all 120 combinations of two to seven members of the pool.

### `voting: estimators`

Voting fits each member independently and combines their predictions. There is no default member list: the full pool is the fixed reference. Different errors may make a member useful, but adding models is not expected to help automatically. The candidates are all 120 combinations of two to seven members of the pool.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
vote_estimators_values = member_values
# One row per value will hold its scores, its seconds and its fitted model.
vote_estimators_rows = []
for value in tqdm(vote_estimators_values, desc="VotingRegressor.estimators", unit="value"):
    started = time.perf_counter()
    candidate = clone(vote_reference).set_params(estimators=fold_owned_members(preparation, value, memory=member_memory), weights=None)
    # The fits of the ten folds and the fit on all the training rows run at the same time; they are scored one by one below.
    fits = joblib.Parallel(n_jobs=4)(
        [joblib.delayed(clone(candidate).fit)(fold["X_fit"], fold["y_fit"]) for fold in combiner_folds]
        + [joblib.delayed(clone(candidate).fit)(X_train, y_train)])
    # The last fit used every training row: it is the model kept for the log and scored on the sealed test rows.
    deployed = fits[-1]
    train_scores = [mean_absolute_error(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(combiner_folds, fits)]
    validation_scores = [mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(combiner_folds, fits)]
    # `configuration` names only the settings that differ from the defaults; `train` and `validation` keep the fold scores for the log.
    vote_estimators_rows.append({"configuration": f"VotingRegressor(estimators=[{', '.join(boost_label if member == 'boosting winner' else member for member, _ in candidate.estimators)}]" + ("" if candidate.weights is None else f", weights={candidate.weights}") + ")", "setting": " + ".join(member for member, _ in value), "seconds": time.perf_counter() - started, "members": len(value), "kinds": len({kinds[member] for member, _ in value}),
        "train_MAE": np.mean(train_scores), "cv_MAE": np.mean(validation_scores),
        "test_MAE": mean_absolute_error(y_test, deployed.predict(X_test)), "train": train_scores,
        "validation": validation_scores, "model": deployed})
# The sweep table: the best five and the worst five of the 120 combinations by validation score, with their sizes, the kinds they mix, the test score and the seconds spent.
vote_estimators_sweep = pd.DataFrame(vote_estimators_rows)
pd.concat([vote_estimators_sweep.nsmallest(5, "cv_MAE"), vote_estimators_sweep.nlargest(5, "cv_MAE")]).set_index("setting")[["members", "kinds", "train_MAE", "cv_MAE", "test_MAE", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
vote_estimators_winner = vote_estimators_sweep.loc[vote_estimators_sweep["cv_MAE"].idxmin()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": vote_estimators_winner["configuration"], "train": vote_estimators_winner["train"], "validation": vote_estimators_winner["validation"], "model": vote_estimators_winner["model"]},
    X_test, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

**The best member subset in this three-fold sweep scores validation MAE 2,170.20 EUR: random forest + boosting winner.** Only membership changes; the other combination settings stay at their declared reference.

__Step 13:__ Count how many kinds of model each voting combination mixes.

In [ ]:
# How many kinds of model each combination mixes, and the validation MAE of the combinations by that number.
vote_by_kinds = vote_estimators_sweep.groupby("kinds")["cv_MAE"].agg(["count", "mean", "min", "max"]).round(4)
display(vote_by_kinds)
# The number of kinds mixed by each of the best five combinations.
print("kinds in the best five:", vote_estimators_sweep.nsmallest(5, "cv_MAE")["kinds"].tolist())

**The number of model kinds does not identify the best subset by itself.** Read each score together with its actual member list; subsets with the same number of kinds can make different errors.

__Step 14:__ Plot the validation MAE of every voting combination against its size and the kinds it mixes.

In [ ]:
# Validation MAE of every combination against the number of members, coloured by how many kinds of model it mixes.
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.stripplot(data=vote_estimators_sweep.astype({"kinds": str}), x="members", y="cv_MAE", hue="kinds",
    palette=sns.blend_palette([PLOT_BLUE, PLOT_ORANGE], n_colors=vote_estimators_sweep["kinds"].nunique()), jitter=0.2, ax=ax)
ax.set(xlabel="members in the combination", ylabel="Validation MAE", title="Voting: every combination of two to seven members")
ax.legend(title="kinds of model")
plt.tight_layout()
plt.show()

Each dot represents one evaluated member subset. Its position and colour describe that subset, so compare the corresponding member names in the ranked table.

__Step 15:__ Change only voting `weights` on the fixed training folds, using equal weights and each vector that doubles one member.

### `voting: weights`

Weights multiply each member's contribution; the default `None` gives equal weight. Emphasising a member is expected to pull the answer towards it. That helps only if its predictions deserve the extra influence. The candidates are equal weights and each emphasis vector that doubles one member while leaving the others at one.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
vote_weights_values = weight_values
# One row per value will hold its scores, its seconds and its fitted model.
vote_weights_rows = []
for value in tqdm(vote_weights_values, desc="VotingRegressor.weights", unit="value"):
    started = time.perf_counter()
    candidate = clone(vote_reference).set_params(weights=value)
    # The fits of the ten folds and the fit on all the training rows run at the same time; they are scored one by one below.
    fits = joblib.Parallel(n_jobs=4)(
        [joblib.delayed(clone(candidate).fit)(fold["X_fit"], fold["y_fit"]) for fold in combiner_folds]
        + [joblib.delayed(clone(candidate).fit)(X_train, y_train)])
    # The last fit used every training row: it is the model kept for the log and scored on the sealed test rows.
    deployed = fits[-1]
    train_scores = [mean_absolute_error(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(combiner_folds, fits)]
    validation_scores = [mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(combiner_folds, fits)]
    # `configuration` names only the settings that differ from the defaults; `train` and `validation` keep the fold scores for the log.
    vote_weights_rows.append({"configuration": f"VotingRegressor(estimators=[{', '.join(boost_label if member == 'boosting winner' else member for member, _ in candidate.estimators)}]" + ("" if candidate.weights is None else f", weights={candidate.weights}") + ")", "setting": "equal" if value is None else "double " + members[value.index(2)][0], "seconds": time.perf_counter() - started,
        "train_MAE": np.mean(train_scores), "cv_MAE": np.mean(validation_scores),
        "test_MAE": mean_absolute_error(y_test, deployed.predict(X_test)), "train": train_scores,
        "validation": validation_scores, "model": deployed})
# The sweep table: one row per value, with the mean fold scores, the test score and the seconds spent.
vote_weights_sweep = pd.DataFrame(vote_weights_rows)
vote_weights_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
vote_weights_winner = vote_weights_sweep.loc[vote_weights_sweep["cv_MAE"].idxmin()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": vote_weights_winner["configuration"], "train": vote_weights_winner["train"], "validation": vote_weights_winner["validation"], "model": vote_weights_winner["model"]},
    X_test, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

**With the full member pool fixed, the best tested weighting scores validation MAE 2,224.02 EUR.** The weight table compares complete weighting vectors on the same rows.

__Step 16:__ Select the best tested voting configuration from the existing sweep rows by validation score.

In [ ]:
vote_complete = pd.DataFrame(vote_estimators_rows + vote_weights_rows)
vote_best = vote_complete.loc[vote_complete["cv_MAE"].idxmin()]
print("best tested voting configuration:", vote_best["configuration"])

**The best complete vote configuration actually evaluated scores validation MAE 2,170.20 EUR.** Its full settings are shown in the table; separate sweep winners are not combined without evaluation.

# <font color='#E8800A'>Stacking</font> <a class="anchor" id="stack"></a>
[Back to TOC](#toc)

A final learner sees member predictions as features. Its training predictions must come from members and preparation that did not fit the predicted row, so stacking cross-fits every training row. Stacking may draw on every algorithm taught so far.

__Step 17:__ Fit a toy final learner to out-of-fold prediction pairs and read its fitted combination.

In [ ]:
toy_meta = np.array([[2., 3.], [4., 4.], [8., 7.]])
toy_meta_response = np.array([3., 5., 8.])
toy_final = BayesianRidge().fit(toy_meta, toy_meta_response)
print("toy member predictions:", toy_meta)
print("toy final predictions:", toy_final.predict(toy_meta).round(3))

**The toy final learner gives 3.0, 5.0 and 8.0 for the three prediction pairs.**

These small arrays stand for predictions made without fitting their own rows; they are not predictions copied from fitted training responses. A same-row construction is forbidden and is not executed. On the course data the helper splits raw rows first, cross-fits preparation and members, trains the final learner on the resulting predictions, and only then refits members for deployment.

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingRegressor.html'>sklearn.ensemble.StackingRegressor(estimators, final_estimator=None, cv=None, passthrough=False, n_jobs=None)</a>

**Definition:**
Fits a final regressor on the out-of-fold predictions of its members, so that the final regressor learns how to combine them.

**Main Parameters:**
- `estimators`: the named members.
- `final_estimator`: the regressor fitted on the member predictions.
- `cv`: the internal folds that produce the out-of-fold predictions.
- `passthrough`: whether the final regressor also sees the original columns.

**Main Attributes (after fitting):**
- `estimators_`: the members refitted on all rows.
- `final_estimator_`: the fitted final regressor.

**Main Methods:**
- `fit(X, y)`: cross-fit the members, fit the final regressor and refit the members.
- `predict(X)`: the final regressor's prediction.
</div>

[Wolpert (1992)](https://doi.org/10.1016/S0893-6080%2805%2980023-1) introduced stacked generalization. [`StackingRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingRegressor.html) and the [user guide section on stacked generalization](https://scikit-learn.org/stable/modules/ensemble.html#stacked-generalization) define the method. Wrapping each member with its preparation gives fold-owned preparation in `StackingRegressor`; the check below matches its out-of-fold predictions and final learner to `FoldStacking` in `preprocessing.py`. The course class caches each member's predictions once per internal fold for reuse across the 120 combinations and combiner search; `StackingRegressor` refits members per combination. Its interface also keeps stacking comparable with blending, which scikit-learn does not implement. Here `passthrough=False` uses member predictions only. When enabled, passed-through features use a separate preprocessor fitted on this stacking fit's training rows, as the class card below explains. Responses are predicted prices, and boosting final learners receive them without preparation.

<div class="alert alert-block alert-info">
<code>FoldStacking(preprocessor, estimators, final_estimator, *, cv=3, method="auto", passthrough=False, random_state=33, memory=None)</code>

A course class in `preprocessing.py`, beside this notebook.

**Definition:**
Cross-fits raw-row members: each internal fold fits members on its fit rows and predicts its score rows. The final learner fits these collected predictions; members are then refitted on all rows.

**Main Parameters:**
- `preprocessor`: the unfitted recipe, fitted inside every member fit on the rows of that fit.
- `estimators`: a list of `(name, learner)` pairs, wrapped by `fold_owned_members`.
- `final_estimator`: the learner fitted on the member predictions.
- `cv`: the number of internal folds; shuffled `StratifiedKFold` for a classifier, `KFold` otherwise.
- `method`: `"auto"` uses the output of `predict`.
- `passthrough`: `False` uses member predictions only. `True` appends features prepared by a separate clone of `preprocessor`, fitted on this stacking fit's training rows and reused at prediction.
- `random_state`: the seed of the internal split.
- `memory`: an optional joblib `Memory` that caches member predictions.

**Main Attributes (after fit):**
- `method_`: the response method used.
- `ownership_`: for every internal fold, the row labels that fitted and the row labels that were scored.
- `meta_estimator_`: the fitted final learner.
- `estimators_`: the members refitted on all the rows.
- `passthrough_preprocessor_`: when enabled, the final learner's fitted feature preparation.

**Main Methods:**
- `fit(X, y)`: cross-fit the members, fit the final learner and refit the members.
- `predict(X)`: the final learner's prediction.

**Differences from scikit-learn:**
[`StackingRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingRegressor.html), described in the [user guide](https://scikit-learn.org/stable/modules/ensemble.html#stacked-generalization), matches the out-of-fold predictions and final learner in the check below, with `passthrough=False` and `predict` responses. `FoldStacking` prepares each internal fit and caches member predictions. Its passed-through features use one training-owned schema; scikit-learn passes through original inputs.
</div>

__Step 18:__ Fit scikit-learn `StackingRegressor` and `FoldStacking` on the same rows and compare their predictions.

In [ ]:
# A small member set and the first fold of raw rows, so the two stackings can be compared quickly.
equal_members = [(name, taught_learners[name]) for name in ["Bayesian ridge", "KNN", "pruned tree, log price"]]
equal_fold = raw_prepared[0]
equal_splitter = KFold(3, shuffle=True, random_state=random_state)
# scikit-learn's stacking: every member is wrapped with the recipe so it prepares its own rows, and the same three-fold splitter produces the out-of-fold predictions.
started = time.perf_counter()
library_stack = StackingRegressor([(name, PreparedEstimator(FoldPreprocessor(recipe), clone(member))) for name, member in equal_members],
    final_estimator=clone(taught_learners["Bayesian ridge"]), cv=equal_splitter, passthrough=False, n_jobs=4).fit(equal_fold["X_fit"], equal_fold["y_fit"])
library_seconds = time.perf_counter() - started
# The course stacking on the same rows, members and splits, with no cache.
started = time.perf_counter()
course_stack = FoldStacking(preparation, equal_members, clone(taught_learners["Bayesian ridge"]), cv=3, random_state=random_state).fit(equal_fold["X_fit"], equal_fold["y_fit"])
course_seconds = time.perf_counter() - started
# The sweeps reuse member predictions through a cache: one fit fills it and the next, timed, fit reads it.
equal_memory = joblib.Memory(".equal_stack_cache", verbose=0)
FoldStacking(preparation, equal_members, clone(taught_learners["Bayesian ridge"]), cv=3, random_state=random_state, memory=equal_memory).fit(equal_fold["X_fit"], equal_fold["y_fit"])
started = time.perf_counter()
cached_stack = FoldStacking(preparation, equal_members, clone(taught_learners["Bayesian ridge"]), cv=3, random_state=random_state, memory=equal_memory).fit(equal_fold["X_fit"], equal_fold["y_fit"])
cached_seconds = time.perf_counter() - started
# Recompute the out-of-fold predictions with `cross_val_predict` under the same splitter and fit the final learner on them: its coefficients must match both stackings.
out_of_fold = np.column_stack([cross_val_predict(PreparedEstimator(FoldPreprocessor(recipe), clone(member)), equal_fold["X_fit"], equal_fold["y_fit"],
    cv=equal_splitter, method="predict") for _, member in tqdm(equal_members, desc='equal members', leave=True, position=0)])
reference_final = clone(taught_learners["Bayesian ridge"]).fit(out_of_fold, equal_fold["y_fit"])
same_out_of_fold = np.array_equal(reference_final.coef_, library_stack.final_estimator_.coef_) and np.array_equal(reference_final.coef_, course_stack.meta_estimator_.coef_)
# Held-out rows go through the refitted members and the final learner of each stacking; they must give the same answers.
library_held_out = library_stack.predict(equal_fold["X_score"])
course_held_out = course_stack.predict(equal_fold["X_score"])
same_held_out = np.array_equal(library_held_out, course_held_out)
# Stop here if the two stackings differ, and print what matched with the three timings.
assert same_out_of_fold and same_held_out, (same_out_of_fold, np.abs(library_held_out - course_held_out).max())
print("out-of-fold predictions give the same final learner in both stacks:", same_out_of_fold)
print("held-out predictions are identical:", same_held_out, "| largest difference", np.abs(library_held_out - course_held_out).max())
print(f"StackingRegressor {library_seconds:.2f} s, FoldStacking {course_seconds:.2f} s, FoldStacking with a warm member cache {cached_seconds:.2f} s")

**The two implementations reproduce the same held-out predictions within the stated tolerance.** The cached version reuses member predictions for the same training and scoring rows; it does not alter the stacking protocol.

<div class="alert alert-block alert-info">
<code>unprepared_boosting(estimator)</code>

A helper in `preprocessing.py`, beside this notebook.

**Definition:**
Copies a boosted learner for prediction columns. A `SampleWeightedPreparedEstimator` base is replaced by its plain learner, retaining `CalibratedClassifierCV` with the same calibration method where used. Other estimators return `clone(estimator)`.

**Main Parameters:**
- `estimator`: the boosted learner to copy.

**Returns:**
An unfitted estimator.

**Differences from scikit-learn:**
`clone` retains a base learner's raw-row preparation. This helper removes it because stacking final learners receive member predictions as columns.
</div>

__Step 19:__ Declare the member combinations and the final learners before any combination score is seen.

Stacking uses the vote's 120 combinations of two to seven pool models and tests the seven pool models as final learners. No test score or combination of single-setting winners enters. The subsets are compared on the first three training folds with Bayesian ridge as final learner. Internal folds stay at 3 because each adds a member refit.

In [ ]:
final_learner_values = list(taught_learners.values()) + [unprepared_boosting(boosting_member)]
stack_reference = FoldStacking(preparation, members, clone(taught_learners["Bayesian ridge"]), cv=3, random_state=random_state, memory=member_memory)

__Step 20:__ Score the FoldStacking reference on the first three training folds.

In [ ]:
fits = joblib.Parallel(n_jobs=4)(joblib.delayed(clone(stack_reference).fit)(fold["X_fit"], fold["y_fit"]) for fold in combiner_folds)
stack_default_train = [mean_absolute_error(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(combiner_folds, fits)]
stack_default_validation = [mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(combiner_folds, fits)]

__Step 21:__ Fit the FoldStacking reference on all the training rows.

In [ ]:
stack_default = clone(stack_reference).fit(X_train, y_train)
stack_default_configuration = f"FoldStacking(estimators=[{', '.join(boost_label if member == 'boosting winner' else member for member, _ in stack_reference.estimators)}], final_estimator={' '.join(str(stack_reference.final_estimator).split())})"

__Step 22:__ Log the FoldStacking reference.

In [ ]:
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": stack_default_configuration,
    "train": stack_default_train, "validation": stack_default_validation, "model": stack_default},
    X_test, y_test, "untuned", metric=mean_absolute_error, label="MAE")

# <font color='#E8800A'>Stacking settings</font> <a class="anchor" id="stack_settings"></a>
[Back to TOC](#toc)

Each sweep changes one setting of the stack and scores every value on the first three training folds, because every combination refits all of its members on each internal fold. The section closes on the best configuration actually evaluated.

| Parameter | Meaning |
| --- | --- |
| `estimators` | Members whose out-of-fold predictions feed the final learner. |
| `final_estimator` | Learner fitted on those predictions. |

[Parameter reference](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingRegressor.html)

__Step 23:__ Change only stacking `estimators` on the fixed training folds, using all 120 combinations of two to seven members of the pool.

### `stacking: estimators`

Out-of-fold member predictions become columns for the final learner. There is no default member list: the full pool is fixed before the sweep, as are the final learner and three internal folds. Extra members are expected to help only when their predictions add useful information. The candidates are all 120 combinations of two to seven members of the pool.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
stack_estimators_values = member_values
# One row per value will hold its scores, its seconds and its fitted model.
stack_estimators_rows = []
for value in tqdm(stack_estimators_values, desc="FoldStacking.estimators", unit="value"):
    started = time.perf_counter()
    candidate = clone(stack_reference).set_params(estimators=value)
    # The fits of the ten folds and the fit on all the training rows run at the same time; they are scored one by one below.
    fits = joblib.Parallel(n_jobs=4)(
        [joblib.delayed(clone(candidate).fit)(fold["X_fit"], fold["y_fit"]) for fold in combiner_folds]
        + [joblib.delayed(clone(candidate).fit)(X_train, y_train)])
    # The last fit used every training row: it is the model kept for the log and scored on the sealed test rows.
    deployed = fits[-1]
    train_scores = [mean_absolute_error(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(combiner_folds, fits)]
    validation_scores = [mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(combiner_folds, fits)]
    # `configuration` names only the settings that differ from the defaults; `train` and `validation` keep the fold scores for the log.
    stack_estimators_rows.append({"configuration": f"FoldStacking(estimators=[{', '.join(boost_label if member == 'boosting winner' else member for member, _ in candidate.estimators)}], final_estimator={' '.join(str(candidate.final_estimator).split())})", "setting": " + ".join(member for member, _ in value), "seconds": time.perf_counter() - started, "members": len(value), "kinds": len({kinds[member] for member, _ in value}),
        "train_MAE": np.mean(train_scores), "cv_MAE": np.mean(validation_scores),
        "test_MAE": mean_absolute_error(y_test, deployed.predict(X_test)), "train": train_scores,
        "validation": validation_scores, "model": deployed})
# The sweep table: the best five and the worst five of the 120 combinations by validation score, with their sizes, the kinds they mix, the test score and the seconds spent.
stack_estimators_sweep = pd.DataFrame(stack_estimators_rows)
pd.concat([stack_estimators_sweep.nsmallest(5, "cv_MAE"), stack_estimators_sweep.nlargest(5, "cv_MAE")]).set_index("setting")[["members", "kinds", "train_MAE", "cv_MAE", "test_MAE", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
stack_estimators_winner = stack_estimators_sweep.loc[stack_estimators_sweep["cv_MAE"].idxmin()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": stack_estimators_winner["configuration"], "train": stack_estimators_winner["train"], "validation": stack_estimators_winner["validation"], "model": stack_estimators_winner["model"]},
    X_test, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

**The best member subset in this three-fold sweep scores validation MAE 2,225.78 EUR: bagged trees + boosting winner.** Only membership changes; the other combination settings stay at their declared reference.

__Step 24:__ Count how many kinds of model each stacking combination mixes.

In [ ]:
# How many kinds of model each combination mixes, and the validation MAE of the combinations by that number.
stack_by_kinds = stack_estimators_sweep.groupby("kinds")["cv_MAE"].agg(["count", "mean", "min", "max"]).round(4)
display(stack_by_kinds)
# The number of kinds mixed by each of the best five combinations.
print("kinds in the best five:", stack_estimators_sweep.nsmallest(5, "cv_MAE")["kinds"].tolist())

**The number of model kinds does not identify the best subset by itself.** Read each score together with its actual member list; subsets with the same number of kinds can make different errors.

__Step 25:__ Plot the validation MAE of every stacking combination against its size and the kinds it mixes.

In [ ]:
# Validation MAE of every combination against the number of members, coloured by how many kinds of model it mixes.
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.stripplot(data=stack_estimators_sweep.astype({"kinds": str}), x="members", y="cv_MAE", hue="kinds",
    palette=sns.blend_palette([PLOT_BLUE, PLOT_ORANGE], n_colors=stack_estimators_sweep["kinds"].nunique()), jitter=0.2, ax=ax)
ax.set(xlabel="members in the combination", ylabel="Validation MAE", title="Stacking: every combination of two to seven members")
ax.legend(title="kinds of model")
plt.tight_layout()
plt.show()

Each dot represents one evaluated member subset. Its position and colour describe that subset, so compare the corresponding member names in the ranked table.

__Step 26:__ Score each stack member by itself on the same three folds, beside the four stacks.

In [ ]:
# Each stack member scored alone on the same three folds. The members reuse the predictions the stack already cached, so nothing is refitted.
stack_member_rows = []
for name, member in tqdm(fold_owned_members(preparation, members, memory=member_memory), desc='stack members', unit='member', leave=True, position=0):
    started = time.perf_counter()
    # Scores on the rows each member was fitted on and on the rows it did not see.
    train_scores, validation_scores = [], []
    for fold in combiner_folds:
        fitted = clone(member).fit(fold["X_fit"], fold["y_fit"])
        train_scores.append(mean_absolute_error(fold["y_fit"], fitted.predict(fold["X_fit"])))
        validation_scores.append(mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"])))
    # The member fitted on all the training rows is scored once on the sealed test rows.
    fitted = clone(member).fit(X_train, y_train)
    stack_member_rows.append({"setting": name, "train_MAE": np.mean(train_scores), "cv_MAE": np.mean(validation_scores),
        "test_MAE": mean_absolute_error(y_test, fitted.predict(X_test)), "seconds": time.perf_counter() - started})
# Best member first, so the table reads against the stack rows above.
stack_member_scores = pd.DataFrame(stack_member_rows).sort_values("cv_MAE")
stack_member_scores.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "seconds"]].round(4)

Compare a stack with its strongest member on the same validation rows. Learning a combination rule does not guarantee an improvement over that member.

__Step 27:__ Change only stacking `final_estimator` on the fixed training folds, using the seven models of the pool.

### `stacking: final_estimator`

The final learner fits out-of-fold member predictions. The default `None` selects scikit-learn's `RidgeCV`; this course reference explicitly fixes the taught Bayesian ridge. A flexible final learner is expected to capture interactions between member predictions but may overfit these few input columns. The candidates are the six earlier models and the boosting winner.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
stack_final_estimator_values = final_learner_values
# One row per value will hold its scores, its seconds and its fitted model.
stack_final_estimator_rows = []
for value in tqdm(stack_final_estimator_values, desc="FoldStacking.final_estimator", unit="value"):
    started = time.perf_counter()
    candidate = clone(stack_reference).set_params(final_estimator=value)
    # The fits of the ten folds and the fit on all the training rows run at the same time; they are scored one by one below.
    fits = joblib.Parallel(n_jobs=4)(
        [joblib.delayed(clone(candidate).fit)(fold["X_fit"], fold["y_fit"]) for fold in combiner_folds]
        + [joblib.delayed(clone(candidate).fit)(X_train, y_train)])
    # The last fit used every training row: it is the model kept for the log and scored on the sealed test rows.
    deployed = fits[-1]
    train_scores = [mean_absolute_error(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(combiner_folds, fits)]
    validation_scores = [mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(combiner_folds, fits)]
    # `configuration` names only the settings that differ from the defaults; `train` and `validation` keep the fold scores for the log.
    stack_final_estimator_rows.append({"configuration": f"FoldStacking(estimators=[{', '.join(boost_label if member == 'boosting winner' else member for member, _ in candidate.estimators)}], final_estimator={' '.join(str(candidate.final_estimator).split())})", "setting": (list(taught_learners) + ["boosting winner"])[final_learner_values.index(value)], "seconds": time.perf_counter() - started,
        "train_MAE": np.mean(train_scores), "cv_MAE": np.mean(validation_scores),
        "test_MAE": mean_absolute_error(y_test, deployed.predict(X_test)), "train": train_scores,
        "validation": validation_scores, "model": deployed})
# The sweep table: one row per value, with the mean fold scores, the test score and the seconds spent.
stack_final_estimator_sweep = pd.DataFrame(stack_final_estimator_rows)
stack_final_estimator_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
stack_final_estimator_winner = stack_final_estimator_sweep.loc[stack_final_estimator_sweep["cv_MAE"].idxmin()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": stack_final_estimator_winner["configuration"], "train": stack_final_estimator_winner["train"], "validation": stack_final_estimator_winner["validation"], "model": stack_final_estimator_winner["model"]},
    X_test, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

**With all members fixed, the best final learner is random forest, scoring validation MAE 2,239.94 EUR.** This comparison does not inherit the member subset selected in the preceding sweep.

__Step 28:__ Select the best tested stacking configuration from the existing sweep rows by validation score.

In [ ]:
stack_complete = pd.DataFrame(stack_estimators_rows + stack_final_estimator_rows)
stack_best = stack_complete.loc[stack_complete["cv_MAE"].idxmin()]
print("best tested stacking configuration:", stack_best["configuration"])

**The best complete stack configuration actually evaluated scores validation MAE 2,225.78 EUR.** Its full settings are shown in the table; separate sweep winners are not combined without evaluation.

# <font color='#E8800A'>Blending</font> <a class="anchor" id="blend"></a>
[Back to TOC](#toc)

Blending reserves one slice of the training rows for the final learner. The members never see that slice, even after deployment, so its predictions are always made on unseen rows.

__Step 29:__ Fit a toy final learner to the predictions made on a held-out slice.

In [ ]:
toy_hold = np.array([[2., 3.], [6., 5.], [9., 8.]])
toy_hold_response = np.array([2.5, 5.5, 8.5])
toy_blender = BayesianRidge().fit(toy_hold, toy_hold_response)
print("held-out member predictions:", toy_hold)
print("blended predictions:", toy_blender.predict(toy_hold).round(3))

**The toy final learner gives 2.5, 5.5 and 8.5 for the three held-out prediction pairs.**

The slice is cut from raw rows before any member or preparation is fitted. Blending uses the same 120 combinations as stacking.

<div class="alert alert-block alert-info">
<code>FoldBlending(preprocessor, estimators, final_estimator, *, fraction=0.25, method="auto", random_state=33, memory=None)</code>

A `FoldStacking` subclass in `preprocessing.py`, beside this notebook.

**Definition:**
The class reserves raw training rows for the final learner. Members fit the remaining rows and predict the held-out rows; the final learner fits those predictions. Members are never refitted on the held-out rows.

**Main Parameters:**
- `preprocessor`: the unfitted recipe, fitted inside every member fit.
- `estimators`: a list of `(name, learner)` pairs, wrapped by `fold_owned_members`.
- `final_estimator`: the learner fitted on the member predictions.
- `fraction`: the share of the training rows held out; the split is stratified for a classifier.
- `method`: `"auto"` uses the output of `predict`.
- `random_state`: the seed of the split.
- `memory`: an optional joblib `Memory` that caches member predictions.

**Main Attributes (after fit):**
- `method_`: the response method used.
- `ownership_`: the row labels that fitted the members and the row labels that were held out.
- `meta_estimator_`: the fitted final learner.
- `estimators_`: the members fitted on the rows that remain.

**Main Methods:**
- `fit(X, y)`: fit the members, then the final learner on the held-out predictions.
- `predict(X)`: the final learner's prediction.

**Differences from scikit-learn:**
This course holdout variant has no scikit-learn estimator. The [user guide section on stacked generalization](https://scikit-learn.org/stable/modules/ensemble.html#stacked-generalization) describes the related stacking method. Stacking trains its final learner on out-of-fold predictions for every row; blending uses one unseen slice. The shared `FoldStacking` interface keeps them comparable.
</div>

__Step 30:__ Construct the blend reference, which holds a quarter of the training rows out for the final learner.

In [ ]:
blend_reference = FoldBlending(preparation, members, clone(taught_learners["Bayesian ridge"]), fraction=0.25, random_state=random_state, memory=member_memory)

__Step 31:__ Score the FoldBlending reference on the first three training folds.

In [ ]:
fits = joblib.Parallel(n_jobs=4)(joblib.delayed(clone(blend_reference).fit)(fold["X_fit"], fold["y_fit"]) for fold in combiner_folds)
blend_default_train = [mean_absolute_error(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(combiner_folds, fits)]
blend_default_validation = [mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(combiner_folds, fits)]

__Step 32:__ Fit the FoldBlending reference on all the training rows.

In [ ]:
blend_default = clone(blend_reference).fit(X_train, y_train)
blend_default_configuration = f"FoldBlending(estimators=[{', '.join(boost_label if member == 'boosting winner' else member for member, _ in blend_reference.estimators)}], final_estimator={' '.join(str(blend_reference.final_estimator).split())}, fraction={blend_reference.fraction})"

__Step 33:__ Log the FoldBlending reference.

In [ ]:
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": blend_default_configuration,
    "train": blend_default_train, "validation": blend_default_validation, "model": blend_default},
    X_test, y_test, "untuned", metric=mean_absolute_error, label="MAE")

# <font color='#E8800A'>Blending settings</font> <a class="anchor" id="blend_settings"></a>
[Back to TOC](#toc)

Each sweep changes one setting of the blend and scores every value on the first three training folds, because every combination refits all of its members. The section closes on the best configuration actually evaluated.

| Parameter | Meaning |
| --- | --- |
| `estimators` | Members fitted on the rows left after the held-out slice. |
| `fraction` | Share of training rows held out for the final learner. |

`FoldBlending` is the course holdout variant in `preprocessing.py`, beside `FoldStacking`. The [user guide section on stacked generalization](https://scikit-learn.org/stable/modules/ensemble.html#stacked-generalization) describes the related stacking method.

__Step 34:__ Change only blending `estimators` on the fixed training folds, using all 120 combinations of two to seven members of the pool.

### `blending: estimators`

Members fit one training subset and predict a separate subset used by the final learner. There is no default member list: the full pool, final learner and holdout fraction form the fixed reference. A new member is expected to help only when it adds information the others lack. The candidates are all 120 combinations of two to seven members of the pool.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
blend_estimators_values = member_values
# One row per value will hold its scores, its seconds and its fitted model.
blend_estimators_rows = []
for value in tqdm(blend_estimators_values, desc="FoldBlending.estimators", unit="value"):
    started = time.perf_counter()
    candidate = clone(blend_reference).set_params(estimators=value)
    # The fits of the ten folds and the fit on all the training rows run at the same time; they are scored one by one below.
    fits = joblib.Parallel(n_jobs=4)(
        [joblib.delayed(clone(candidate).fit)(fold["X_fit"], fold["y_fit"]) for fold in combiner_folds]
        + [joblib.delayed(clone(candidate).fit)(X_train, y_train)])
    # The last fit used every training row: it is the model kept for the log and scored on the sealed test rows.
    deployed = fits[-1]
    train_scores = [mean_absolute_error(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(combiner_folds, fits)]
    validation_scores = [mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(combiner_folds, fits)]
    # `configuration` names only the settings that differ from the defaults; `train` and `validation` keep the fold scores for the log.
    blend_estimators_rows.append({"configuration": f"FoldBlending(estimators=[{', '.join(boost_label if member == 'boosting winner' else member for member, _ in candidate.estimators)}], final_estimator={' '.join(str(candidate.final_estimator).split())}, fraction={candidate.fraction})", "setting": " + ".join(member for member, _ in value), "seconds": time.perf_counter() - started, "members": len(value), "kinds": len({kinds[member] for member, _ in value}),
        "train_MAE": np.mean(train_scores), "cv_MAE": np.mean(validation_scores),
        "test_MAE": mean_absolute_error(y_test, deployed.predict(X_test)), "train": train_scores,
        "validation": validation_scores, "model": deployed})
# The sweep table: the best five and the worst five of the 120 combinations by validation score, with their sizes, the kinds they mix, the test score and the seconds spent.
blend_estimators_sweep = pd.DataFrame(blend_estimators_rows)
pd.concat([blend_estimators_sweep.nsmallest(5, "cv_MAE"), blend_estimators_sweep.nlargest(5, "cv_MAE")]).set_index("setting")[["members", "kinds", "train_MAE", "cv_MAE", "test_MAE", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
blend_estimators_winner = blend_estimators_sweep.loc[blend_estimators_sweep["cv_MAE"].idxmin()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": blend_estimators_winner["configuration"], "train": blend_estimators_winner["train"], "validation": blend_estimators_winner["validation"], "model": blend_estimators_winner["model"]},
    X_test, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

**The best member subset in this three-fold sweep scores validation MAE 2,303.79 EUR: pruned tree, log price + bagged trees + random forest + boosting winner.** Only membership changes; the other combination settings stay at their declared reference.

__Step 35:__ Count how many kinds of model each blending combination mixes.

In [ ]:
# How many kinds of model each combination mixes, and the validation MAE of the combinations by that number.
blend_by_kinds = blend_estimators_sweep.groupby("kinds")["cv_MAE"].agg(["count", "mean", "min", "max"]).round(4)
display(blend_by_kinds)
# The number of kinds mixed by each of the best five combinations.
print("kinds in the best five:", blend_estimators_sweep.nsmallest(5, "cv_MAE")["kinds"].tolist())

**The number of model kinds does not identify the best subset by itself.** Read each score together with its actual member list; subsets with the same number of kinds can make different errors.

__Step 36:__ Plot the validation MAE of every blending combination against its size and the kinds it mixes.

In [ ]:
# Validation MAE of every combination against the number of members, coloured by how many kinds of model it mixes.
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.stripplot(data=blend_estimators_sweep.astype({"kinds": str}), x="members", y="cv_MAE", hue="kinds",
    palette=sns.blend_palette([PLOT_BLUE, PLOT_ORANGE], n_colors=blend_estimators_sweep["kinds"].nunique()), jitter=0.2, ax=ax)
ax.set(xlabel="members in the combination", ylabel="Validation MAE", title="Blending: every combination of two to seven members")
ax.legend(title="kinds of model")
plt.tight_layout()
plt.show()

Each dot represents one evaluated member subset. Its position and colour describe that subset, so compare the corresponding member names in the ranked table.

__Step 37:__ Change only blending `fraction` on the fixed training folds, with the sweep list `[0.2, 0.25, 0.33]`.

### `blending: fraction`

This fraction of training rows is reserved for fitting the final learner. The course class defaults to `0.25` and has no scikit-learn equivalent. A larger fraction is expected to improve estimation of the combining rule while weakening members trained on fewer rows. The candidate list is `[0.2, 0.25, 0.33]`.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
blend_fraction_values = [0.2, 0.25, 0.33]
# One row per value will hold its scores, its seconds and its fitted model.
blend_fraction_rows = []
for value in tqdm(blend_fraction_values, desc="FoldBlending.fraction", unit="value"):
    started = time.perf_counter()
    candidate = clone(blend_reference).set_params(fraction=value)
    # The fits of the ten folds and the fit on all the training rows run at the same time; they are scored one by one below.
    fits = joblib.Parallel(n_jobs=4)(
        [joblib.delayed(clone(candidate).fit)(fold["X_fit"], fold["y_fit"]) for fold in combiner_folds]
        + [joblib.delayed(clone(candidate).fit)(X_train, y_train)])
    # The last fit used every training row: it is the model kept for the log and scored on the sealed test rows.
    deployed = fits[-1]
    train_scores = [mean_absolute_error(fold["y_fit"], fitted.predict(fold["X_fit"])) for fold, fitted in zip(combiner_folds, fits)]
    validation_scores = [mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"])) for fold, fitted in zip(combiner_folds, fits)]
    # `configuration` names only the settings that differ from the defaults; `train` and `validation` keep the fold scores for the log.
    blend_fraction_rows.append({"configuration": f"FoldBlending(estimators=[{', '.join(boost_label if member == 'boosting winner' else member for member, _ in candidate.estimators)}], final_estimator={' '.join(str(candidate.final_estimator).split())}, fraction={candidate.fraction})", "setting": value, "seconds": time.perf_counter() - started,
        "train_MAE": np.mean(train_scores), "cv_MAE": np.mean(validation_scores),
        "test_MAE": mean_absolute_error(y_test, deployed.predict(X_test)), "train": train_scores,
        "validation": validation_scores, "model": deployed})
# The sweep table: one row per value, with the mean fold scores, the test score and the seconds spent.
blend_fraction_sweep = pd.DataFrame(blend_fraction_rows)
blend_fraction_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
blend_fraction_winner = blend_fraction_sweep.loc[blend_fraction_sweep["cv_MAE"].idxmin()]
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": blend_fraction_winner["configuration"], "train": blend_fraction_winner["train"], "validation": blend_fraction_winner["validation"], "model": blend_fraction_winner["model"]},
    X_test, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

**With the full member pool fixed, fraction 0.2 gives the lowest tested validation MAE, 2,372.55 EUR.** The other settings remain unchanged.

__Step 38:__ Select the best tested blending configuration from the existing sweep rows by validation score.

In [ ]:
blend_complete = pd.DataFrame(blend_estimators_rows + blend_fraction_rows)
blend_best = blend_complete.loc[blend_complete["cv_MAE"].idxmin()]
print("best tested blending configuration:", blend_best["configuration"])

**The best complete blend configuration actually evaluated scores validation MAE 2,303.79 EUR.** Its full settings are shown in the table; separate sweep winners are not combined without evaluation.

__Step 39:__ Audit the stack and blend row ownership from the two default fits.

In [ ]:
for label_owned, construction in [("stack", stack_default), ("blend", blend_default)]:
    overlaps = [len(set(a) & set(b)) for a, b in construction.ownership_]
    print(label_owned, "internal fit/score overlap counts:", overlaps)
    assert not any(overlaps)
print("stack meta-training coverage:", sum(len(b) for a, b in stack_default.ownership_))
print("blend member rows:", len(blend_default.ownership_[0][0]), "meta rows:", len(blend_default.ownership_[0][1]))

# <font color='#E8800A'>Selection of the combiners</font> <a class="anchor" id="sel2"></a>
[Back to TOC](#toc)

The combiner search tests voting, stacking and blending over the pool subsets and swept settings. Nested cross-validation evaluates this selection first. In each of its ten outer folds, the boosting member uses settings selected on that fold's training rows. The final search uses all training rows and the final boosting winner.

The grid covers voting, stacking and blending: every combination of two to seven members and every swept setting, with the weights only where the members are the full seven. The stored winning configurations of the earlier models are fixed teaching inputs, not a fresh outer-fold search over their original tuning histories. As before, the search evaluates 60 candidates, drawn uniformly without replacement from the enumerated space with a seed of 33 before any score is seen, and the same 60 feed the nested estimate and the search on all the training rows.

__Step 40:__ Declare the conditional grids of the combiners, which fix the search space before any search score exists.

In [ ]:
# The search space of the second selection: voting, stacking and blending over the 120 combinations of two to seven pool models and their swept settings, with the weights only on the full pool of seven.
# A member is a position in the pool, so that each outer fold can put its own boosting winner in the last place.
combiner_grid = [
    # Voting: every member combination, and separately the weights, which only fit the full pool; its members prepare their own rows.
    {"preprocessor": [FunctionTransformer()], "estimator": [clone(vote_reference)],
     "estimator__estimators": member_index},
    {"preprocessor": [FunctionTransformer()], "estimator": [clone(vote_reference)],
     "estimator__estimators": [tuple(range(len(members)))],
     "estimator__weights": vote_weights_values},
    # Stacking: every member combination with every final learner, given as a position in the pool.
    {"preprocessor": [FunctionTransformer(kw_args={})], "estimator": [clone(stack_reference)],
     "estimator__estimators": member_index,
     "estimator__final_estimator": list(range(len(final_learner_values)))},
    # Blending: every member combination with every held-out fraction.
    {"preprocessor": [FunctionTransformer(inv_kw_args={})], "estimator": [clone(blend_reference)],
     "estimator__estimators": member_index,
     "estimator__fraction": blend_fraction_values}
]

__Step 41:__ Enumerate every valid combination of the combiners, and draw 60 of them without replacement before any score is seen.

In [ ]:
# The conditional grids hold every valid combination; the search draws 60 of them, uniformly and without replacement, before any score exists.
print("combinations by group:", [len(ParameterGrid(group)) for group in combiner_grid])
print("combinations in the space:", len(ParameterGrid(combiner_grid)))

**The conditional grids of the combiners hold 1328 valid combinations, and the search evaluates 60 of them.**

__Step 42:__ Give each outer fold its own boosting member, and turn the drawn combinations into the candidates of that fold.

In [ ]:
# One boosting member for each outer fold, from the settings that fold's search chose in the first selection, and one for the search on all the training rows.
boost_members = [clone(chosen["estimator"]).set_params(**{key.split("estimator__", 1)[1]: value for key, value in chosen.items() if key.startswith("estimator__")})
    for chosen in boost_fold_settings + [boost_settings]]
# For each of them the grid keeps its shape: a position in the pool becomes the model at that position, with that fold's boosting member last.
combiner_grids = []
for boost_member in boost_members:
    pool = list(taught_learners.items()) + [("boosting winner", boost_member)]
    finals = list(taught_learners.values()) + [unprepared_boosting(boost_member)]
    fold_grid = []
    for group in combiner_grid:
        # A vote takes members that prepare their own rows; stacking and blending prepare inside their own splits and take the plain models.
        voting = type(group["estimator"][0]).__name__.startswith("Voting")
        fold_grid.append({name: ([fold_owned_members(preparation, [pool[i] for i in value], memory=member_memory) if voting else [pool[i] for i in value] for value in values]
                                 if name == "estimator__estimators" else [finals[value] for value in values] if name == "estimator__final_estimator" else values)
                          for name, values in group.items()})
    combiner_grids.append(fold_grid)

__Step 43:__ Wrap each grid and the preprocessing in one search, with the same three inner folds.

In [ ]:
combiner_searches = [CachedSearchCV(RandomizedSearchCV(PreparedEstimator(FunctionTransformer(), clone(vote_reference)), grid, n_iter=60, random_state=random_state, cv=inner, scoring="neg_mean_absolute_error", n_jobs=4)) for grid in combiner_grids]

__Step 44:__ Estimate the whole search with the ten outer folds of the training partition, returning scores rather than models.

In [ ]:
combiner_nested_started = time.perf_counter()
# Each of the ten outer folds runs its own search, with that fold's boosting winner as the boosting member, at most two outer searches at once; each inner search uses four workers.
outer_runs = joblib.Parallel(n_jobs=2, return_as="generator")(
    joblib.delayed(cross_validate)(combiner_searches[position], X_train, y_train, cv=[split],
        scoring="neg_mean_absolute_error", return_train_score=True, n_jobs=1, error_score="raise")
    for position, split in enumerate(folds.split(X_train, y_train)))
# Collect the outer results as the workers finish; the progress bar fills in bursts.
combiner_fold_results = list(tqdm(outer_runs, total=folds.get_n_splits(),
                                  desc="Nested CV, combiners: outer folds", unit="fold", leave=True, position=0))
# The outer validation scores estimate the whole procedure, selection step included; the training scores sit beside them.
combiner_nested_scores = {key: np.concatenate([result[key] for result in combiner_fold_results]) for key in ("train_score", "test_score")}
combiner_nested_validation = -combiner_nested_scores["test_score"]
combiner_nested_train = -combiner_nested_scores["train_score"]
combiner_nested_seconds = time.perf_counter() - combiner_nested_started
# Report the estimate, its spread over the folds and the time.
print(f"nested validation MAE: {combiner_nested_validation.mean():.4f}")
print(f"fold SD: {combiner_nested_validation.std(ddof=1):.4f}")
print(f"nested wall time: {combiner_nested_seconds:.2f} seconds")

**The complete search scores a nested validation MAE of 2,208.81 EUR with a fold SD of 141.88.** The space holds 1328 valid combinations, 60 of them drawn for the search.

# <font color='#E8800A'>The combiner winner</font> <a class="anchor" id="combiner_winner"></a>
[Back to TOC](#toc)

With the search space fixed and its nested estimate available, the same candidates are fitted to all the training rows, with the boosting winner of the first selection as the boosting member. Its best estimator is the combiner winner.

__Step 45:__ Run the declared grid of the combiners on all the training rows and keep its winner.

In [ ]:
combiner_started = time.perf_counter()
combiner_search = combiner_searches[-1].fit(X_train, y_train)
combiner_winner = combiner_search.best_estimator_
combiner_seconds = time.perf_counter() - combiner_started
print("chosen parameters:", combiner_search.best_params_)
print("winner own inner-CV score:", combiner_search.best_score_)
print("combinations evaluated:", len(combiner_search.cv_results_["params"]))
print(f"search wall time: {combiner_seconds:.2f} seconds")

__Step 46:__ Score the combiner winner on the ten outer folds, and log it beside its test score.

In [ ]:
# Score the winner on the same folds: stacking and blending split their own rows, so each fold's raw training rows go to the combiner.
combiner_train, combiner_validation = [], []
for fold in tqdm(raw_prepared, desc='raw folds', leave=True, position=0):
    fitted = clone(combiner_winner.estimator).fit(fold["X_fit"], fold["y_fit"])
    combiner_train.append(mean_absolute_error(fold["y_fit"], fitted.predict(fold["X_fit"])))
    combiner_validation.append(mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"])))

In [ ]:
combiner_settings = combiner_search.best_params_
combiner_constructor = clone(combiner_settings["estimator"]).set_params(**{key.split("estimator__", 1)[1]: value for key, value in combiner_settings.items() if key.startswith("estimator__")})
# Name the winner by its members and the settings that differ from the defaults.
combiner_label = (f"{type(combiner_constructor).__name__}(estimators=[{', '.join(boost_label if member == 'boosting winner' else member for member, _ in combiner_constructor.estimators)}]"
    + "".join(f", {key.split('estimator__', 1)[1]}={' '.join(str(value).split())}" for key, value in combiner_settings.items()
                if key.startswith("estimator__") and key != "estimator__estimators") + ")")
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": combiner_label, "train": combiner_train, "validation": combiner_validation, "model": combiner_winner},
    X_test, y_test, "winner's own score", metric=mean_absolute_error, label="MAE")

__Step 47:__ Log the combiner winner with its nested estimate as the validation score.

In [ ]:
record(model_log, {"frame": type(folds).__name__, "rows": len(y_train), "folds": folds.get_n_splits(),
    "name": f"combiner search over {len(combiner_search.cv_results_['params'])} combinations", "train": combiner_nested_train, "validation": combiner_nested_validation, "model": combiner_winner},
    X_test, y_test, "nested estimate", metric=mean_absolute_error, label="MAE")

__Step 48:__ Read the test score of the combiner winner from the log.

In [ ]:
combiner_test = model_log[-1]["test MAE"]

**The search chose an equal-weight average of the forest and the boosting winner, with held-out MAE 2,192.92 EUR and an inner-CV score of 2,205.92.** Its own score over the ten outer folds is 2,188.13, against a nested estimate of 2,208.81 for the whole search.

# <font color='#E8800A'>The course board</font> <a class="anchor" id="board"></a>
[Back to TOC](#toc)

The nested estimate describes the declared selection procedure. The test score belongs to the actual final-grid winner. Earlier results join the board only when their data and protocol fingerprints match.

__Step 49:__ Build the Dummy floor on training rows, then join the saved earlier results.

In [ ]:
board = Scoreboard("cars4you price", "MAE (EUR)", context=result_context(
    "../../data/interim/cars4you.csv", recipe_log, y_train.index.tolist(), y_test.index.tolist(),
    outer_folds=outer_folds(folds, X_train, y_train)),
    floor=mean_absolute_error(y_test, DummyRegressor(strategy="mean").fit(X_train, y_train).predict(X_test)))
for week, earlier_model, earlier_cv, earlier_test in results_before(11, "regression", results_dir, metric=board.metric, context=board.context):
    board.record(week, earlier_model, earlier_cv, earlier_test)

__Step 50:__ Save the nested estimate, the winner test score and its complete constructor settings.

In [ ]:
# The week's row is the selection with the better nested estimate; the winner of the other selection stays in the log.
selections = {"boosting": {"nested": boost_nested_validation, "test": boost_test, "label": boost_label, "model": boosting_winner},
    "combiner": {"nested": combiner_nested_validation, "test": combiner_test, "label": combiner_label, "model": combiner_winner}}
board_selection = "combiner" if combiner_nested_validation.mean() < boost_nested_validation.mean() else "boosting"
board_nested = selections[board_selection]["nested"]
board_test = selections[board_selection]["test"]
chosen_label = selections[board_selection]["label"]
chosen_model = selections[board_selection]["model"]
print("Boosting nested MAE: %.4f" % np.mean(boost_nested_validation))
print("Combining nested MAE: %.4f" % np.mean(combiner_nested_validation))
print("board selection:", board_selection)
# Fixed settings and their own family reference use the same course folds.
if board_selection == "boosting":
    tuning_comparison = selection["tuning_comparison"]
else:
    selected_family = type(chosen_model.estimator).__name__
    reference_row = next(row for row in model_log
        if row.get("validation MAE is") == "untuned" and row["model"].startswith(selected_family + "("))
    candidate_row = next(row for row in model_log
        if row.get("validation MAE is") == "winner's own score" and row["model"] == chosen_label)
    tuning_comparison = dict(
        protocol="fixed_settings_on_course_folds", family=selected_family,
        candidate_model=chosen_label, reference_model=reference_row["model"],
        reference_kind="declared_reference", candidate_cv=float(candidate_row["validation MAE"]),
        reference_cv=float(reference_row["validation MAE"]),
        candidate_test=float(candidate_row["test MAE"]), reference_test=float(reference_row["test MAE"]))
comparison_reference_cv = tuning_comparison["reference_cv"]
comparison_reference_test = tuning_comparison["reference_test"]
# Record the nested estimate as the cross-validated number and the held-out score as the test number; save the row with its baseline and the fitted winner.
board.record(11, chosen_label, round(board_nested.mean(), 4), round(board_test, 4))
board.save(results_dir, 11, "regression", baseline_cv=round(comparison_reference_cv, 4), baseline_test=round(comparison_reference_test, 4))
joblib.dump(chosen_model, f"{results_dir}/week_11_regression_model.joblib")
# Print the board with every earlier result and the no-model floor.
print(board.table())
# Preserve both the nested procedure estimate and the fixed-setting comparison.
result_path = Path(results_dir) / "week_11.json"
result_document = json.loads(result_path.read_text(encoding="utf-8"))
result_document["regression"]["tuning_comparison"] = tuning_comparison
result_path.write_text(json.dumps(result_document, indent=1) + "\n", encoding="utf-8")


**The week's row takes the combiner selection, whose nested estimate of 2,208.81 EUR is lower than the 2,237.74 of the boosting selection.** The row scores 2,208.81 EUR cross-validated and 2,192.92 held out, against 2,366.26 and 2,345.30 for the forest row above, so the combiner winner improves on the forest here. The no-model floor is 6,843.46 EUR.

# <font color='#E8800A'>Key takeaways</font> <a class="anchor" id="takeaways"></a>
[Back to TOC](#toc)

1. Voting combines predictions with a fixed rule; stacking learns its rule from out-of-fold predictions.
2. Blending reserves training rows for the final learner, reducing the data available to its members.
3. Member lists and final learners are explicit fixed references: each isolated sweep changes one setting.
4. Each outer combiner search uses the boosting settings selected on that outer training fold.
5. The overall Week 11 row compares the two nested selection procedures.

# <font color='#E8800A'>Exit ticket</font> <a class="anchor" id="exit"></a>
[Back to TOC](#toc)

The boosting search has nested validation MAE 2,237.74 EUR and the combining search 2,208.81 EUR. Why must each outer combining fold use its own boosting selection rather than the winner fitted on all training rows?

**Answer.** The boosting member must be selected using only that outer fold's training rows. Loading the winner selected on all training rows would let the outer validation rows influence the member selection.

# <font color='#E8800A'>References</font> <a class="anchor" id="references"></a>
[Back to TOC](#toc)

Original papers support the method explanations; library documentation defines the estimator contracts.

- [scikit-learn ensemble methods](https://scikit-learn.org/stable/modules/ensemble.html)
- [VotingRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.VotingRegressor.html)
- [StackingRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingRegressor.html)
- [Wolpert, D. H. (1992). Stacked Generalization. Neural Networks, 5(2), 241-259.](https://doi.org/10.1016/S0893-6080%2805%2980023-1)
- [Bergstra, J. and Bengio, Y. (2012). Random Search for Hyper-Parameter Optimization. Journal of Machine Learning Research, 13(10), 281-305.](https://jmlr.org/papers/v13/bergstra12a.html)
- [RandomizedSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html)
- [Nested versus non-nested cross-validation](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html)
- [joblib disk-based caching](https://joblib.readthedocs.io/en/latest/user_guide/memory.html)
- [joblib serialization and loading](https://joblib.readthedocs.io/en/latest/user_guide/persistence.html)